## 복습문제

### 문제 1. API 키를 `.env` 파일에 보관하는 이유는 무엇인가요? 
<details>
<summary>정답 보기</summary>  

- **보안성**: 키 하드코딩/깃 커밋 방지 → 외부 유출 리스크 감소
- **환경 분리**: 로컬/스테이징/프로덕션 별로 키 교체 용이
- **자동화**: CI/CD에서 시크릿 관리와 연동 쉬움
</details>

In [ ]:
### 여기에 정답을 작성해보세요

### 문제 2. **프롬프트 설계**: 

"너는 용접 전문가야"라는 시스템 프롬프트를 넣고, 사용자가 배관 부식 원인을 물었을 때 기술적인 답변을 출력하는 체인을 작성해보세요.  

<details>
<summary>정답 보기</summary>

```python 
from dotenv import load_dotenv              # .env 파일에서 환경변수(OPENAI_API_KEY) 불러오기
import os                                   # OS 환경변수 접근
from langchain_core.prompts import ChatPromptTemplate  # 프롬프트 템플릿(변수 삽입/재사용)
from langchain_openai import ChatOpenAI                # OpenAI 대화형 LLM 래퍼
from langchain_core.output_parsers import StrOutputParser  # 모델 응답을 문자열로 파싱

# ===========================================
# 2) 환경변수 로드
# ===========================================
load_dotenv()                               # 현재 작업 디렉토리의 .env를 로드
# os.getenv("OPENAI_API_KEY")               # 내부적으로 ChatOpenAI가 환경변수를 읽음

# ===========================================
# 3) 프롬프트 템플릿 정의
#    - System: 모델 역할을 '용접 전문가'로 고정
#    - User: 실습 시 입력될 질문 자리표시자 {question}
# ===========================================
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 용접 전문가이자 재료공학 엔지니어야. 한국어로 간결하지만 정확하게 답변해. "
               "표준/코드(AWS/ASME/KS)나 공정 용어는 정확히 사용해."),
    ("user", "{question}")
])

# ===========================================
# 4) 모델 & 파서 준비
# ===========================================
model = ChatOpenAI(model="gpt-3.5-turbo")   # 필요 시 gpt-4o 등으로 교체 가능
parser = StrOutputParser()                   # 메시지를 최종 문자열로 변환

# ===========================================
# 5) LCEL로 체인 구성: Prompt -> Model -> Parser
# ===========================================
chain = prompt | model | parser

# ===========================================
# 6) 실행 예시
# ===========================================
question = "SUS304 배관에서 담수 조건 핀홀 부식 원인을 설명해줘. 예방책도 제시해줘."
answer = chain.invoke({"question": question})  # 자리표시자에 question 바인딩
print(answer)                                  # 최종 텍스트 출력
```
</details>

In [ ]:
### 여기에 정답을 작성해보세요
from dotenv import load_dotenv              # .env 파일에서 환경변수(OPENAI_API_KEY) 불러오기
import os                                   # OS 환경변수 접근
from langchain_core.prompts import ChatPromptTemplate  # 프롬프트 템플릿(변수 삽입/재사용)
from langchain_openai import ChatOpenAI                # OpenAI 대화형 LLM 래퍼
from langchain_core.output_parsers import StrOutputParser  # 모델 응답을 문자열로 파싱

### 문제 3. **대화 기억**: 

사용자의 첫 질문에 "용접의 종류"를 답변한 뒤, 두 번째 질문에 "방금 말한 종류 중 하나를 설명해"라고 했을 때  
올바르게 이어질 수 있도록 메모리를 적용해보세요. 

<details>
<summary>정답 보기</summary>

```python 
from dotenv import load_dotenv
import os
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # 히스토리용 플레이스홀더
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# ===========================================
# 2) 환경변수 로드
# ===========================================
load_dotenv()

# ===========================================
# 3) 대화 히스토리 컨테이너 (간단 리스트)
#    - (role, content) 튜플 목록으로 관리
#    - 실제 프로덕션에선 ConversationBufferMemory 등을 사용할 수 있음
# ===========================================
history = []  # 예: [("user","..."), ("assistant","..."), ...]

# ===========================================
# 4) 프롬프트 템플릿
#    - MessagesPlaceholder("history")를 이용해 멀티턴 히스토리 삽입
# ===========================================
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 용접 전문가야. 한국어로 단계적으로 설명해."),
    MessagesPlaceholder(variable_name="history"),  # 여기에 이전 턴들 삽입
    ("user", "{input}")                             # 현재 사용자 질문
])

# ===========================================
# 5) 모델 & 파서 & 체인 구성
# ===========================================
model = ChatOpenAI(model="gpt-3.5-turbo")
parser = StrOutputParser()
chain = prompt | model | parser

# ===========================================
# 6) 1차 질문/응답
# ===========================================
user_q1 = "용접의 대표적인 공정 3가지를 요약해줘."
assistant_a1 = chain.invoke({"history": history, "input": user_q1})
print("Assistant(1):", assistant_a1)

# 히스토리 업데이트 (다음 턴에 맥락으로 사용)
history.append(("user", user_q1))
history.append(("assistant", assistant_a1))

# ===========================================
# 7) 2차 질문/응답 (이전 답을 참조하도록 유도)
# ===========================================
user_q2 = "방금 말한 3가지 중 FCAW를 장단점 중심으로 자세히 설명해줘."
assistant_a2 = chain.invoke({"history": history, "input": user_q2})
print("Assistant(2):", assistant_a2)

# 히스토리 업데이트(선택)
history.append(("user", user_q2))
history.append(("assistant", assistant_a2))

```
</details>

In [1]:
### 여기에 정답을 작성해보세요
from dotenv import load_dotenv
import os
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

### 문제 4. **문서 기반 챗봇 확장**: FAISS에 여러 PDF 문서를 임포트해 검색 가능한 챗봇을 만들어보세요.

<details>
<summary>정답 보기</summary>

```python 
# ===========================================
# 1) 라이브러리 임포트
# ===========================================
from dotenv import load_dotenv
import os
from langchain.text_splitter import CharacterTextSplitter        # 문서 청크 분할
from langchain_community.vectorstores import FAISS               # 벡터DB로 FAISS 사용
from langchain_openai import OpenAIEmbeddings                    # OpenAI 임베딩
from langchain_core.prompts import ChatPromptTemplate            # RAG 프롬프트
from langchain_openai import ChatOpenAI                          # LLM
from langchain_core.output_parsers import StrOutputParser        # 출력 파서

# (선택) PDF 로더를 쓰려면 pypdf 설치 후 아래 임포트 가능
# from langchain_community.document_loaders import PyPDFLoader   # uv add pypdf

# ===========================================
# 2) 환경변수 로드
# ===========================================
load_dotenv()

# ===========================================
# 3) 원문 수집
#    - 간단히 텍스트 리스트로 예시
#    - PDF 사용 시: loader = PyPDFLoader("file.pdf"); docs = loader.load()
# ===========================================
raw_texts = [
    "SUS304는 오스테나이트계 스테인리스로 내식성이 우수하나 염화이온 환경에선 틈부식 위험이 존재한다.",
    "FCAW(Flux-Cored Arc Welding)는 고생산성 공정으로 야외 시공에서 이점이 있지만 슬래그 제거가 필요하다.",
    "ASME Section IX는 용접 절차 및 성능시험에 대한 요건을 규정한다."
]

# ===========================================
# 4) 텍스트 → Document로 변환 및 분할
# ===========================================
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
documents = splitter.create_documents(raw_texts)  # 리스트[str] → 리스트[Document]

# ===========================================
# 5) 임베딩 & 벡터DB 색인
# ===========================================
embeddings = OpenAIEmbeddings()           # OpenAI 임베딩 사용
vectordb = FAISS.from_documents(documents, embeddings)  # 인메모리 FAISS 인덱스 생성
retriever = vectordb.as_retriever(search_kwargs={"k": 3})  # 상위 3개 유사 청크 검색

# ===========================================
# 6) 간단한 RAG 프롬프트 & LLM
#    - 검색 결과를 context로 주입해 정확한 답을 유도
# ===========================================
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 재료/용접 전문가야. 아래 문맥(context)을 근거로 질문에 답하되, "
     "모르면 모른다고 말해. 추측하지 마라.\n\n[CONTEXT]\n{context}\n"),
    ("user", "{question}")
])
llm = ChatOpenAI(model="gpt-3.5-turbo")
parser = StrOutputParser()

# ===========================================
# 7) RAG 질의 함수
# ===========================================
def rag_answer(question: str) -> str:
    # (1) 검색
    docs = retriever.get_relevant_documents(question)
    context = "\n\n".join([d.page_content for d in docs])
    # (2) 생성
    chain = rag_prompt | llm | parser
    return chain.invoke({"context": context, "question": question})

# ===========================================
# 8) 실행 예시
# ===========================================
q = "야외 시공에서 FCAW의 장점은 뭐야? 틈부식 관련 주의사항도 알려줘."
print(rag_answer(q))

```
</details>

In [3]:
### 여기에 정답을 작성해보세요
from dotenv import load_dotenv
import os
from langchain_text_splitters import CharacterTextSplitter        # 문서 청크 분할
from langchain_community.vectorstores import FAISS               # 벡터DB로 FAISS 사용
from langchain_openai import OpenAIEmbeddings                    # OpenAI 임베딩
from langchain_core.prompts import ChatPromptTemplate            # RAG 프롬프트
from langchain_openai import ChatOpenAI                          # LLM
from langchain_core.output_parsers import StrOutputParser        # 출력 파서